In [0]:
from pyspark.sql import functions as F

df_orders = spark.read.table("workspace.restaurantdb.orders")
df_details = spark.read.table("workspace.restaurantdb.order_details")
df_menu = spark.read.table("workspace.restaurantdb.menu")
df_employees = spark.read.table("workspace.restaurantdb.employees")

#Exploring Orders table

In [0]:
df_orders.printSchema()

In [0]:
df_orders.show(5)

In [0]:
display(df_orders)

In [0]:
df_orders.count()

In [0]:
# 1. Extraire le nom du jour et le numéro du jour (pour le tri)
df_days = df_orders.withColumn("day_name", F.date_format("timestamp", "EEEE")) \
                   .withColumn("day_number", F.dayofweek("timestamp"))

# 2. Compter le nombre de commandes par jour
orders_by_day = df_days.groupBy("day_number", "day_name") \
                       .agg(F.count("order_id").alias("nb_commandes")) \
                       .orderBy("day_number")

display(orders_by_day)

Databricks visualization. Run in Databricks to view.

In [0]:
target_date = "2025-09-15"

# 2. Filtrer, extraire l'heure et compter
hourly_orders = df_orders.filter(F.to_date("timestamp") == target_date) \
    .withColumn("hour", F.hour("timestamp")) \
    .groupBy("hour") \
    .agg(F.count("order_id").alias("nb_commandes")) \
    .orderBy("hour")

display(hourly_orders)

Databricks visualization. Run in Databricks to view.

In [0]:
df_weather = spark.read.table("workspace.restaurantdb.weather")

# 2. Joindre les tables sur weather_id
df_combined = df_orders.join(df_weather, "weather_id")

# 3. Créer une colonne binaire : 'Rainy' vs 'Other'
df_analysis = df_combined.withColumn(
    "weather_group", 
    F.when(F.col("condition").isin("rainy", "stormy"), "Bad weather").otherwise("Clear/Other")
)

# 4. Compter par groupe météo et par type de commande
result = df_analysis.groupBy("weather_group", "order_type") \
    .agg(F.count("order_id").alias("nb_commandes")) \
    .orderBy("weather_group", "order_type")

display(result)

Databricks visualization. Run in Databricks to view.

#Exploring Employees table

In [0]:
display(df_employees)

In [0]:
# 2. Calculer le montant par ligne de commande (qty * price)
df_sales = df_details.join(df_menu, "item_id") \
                     .withColumn("line_total", F.col("qty") * F.col("price"))

# 3. Joindre avec les commandes et les employés
df_final = df_sales.join(df_orders, "order_id") \
                   .join(df_employees, df_orders.server_id == df_employees.emp_id)

# 4. Aggréger par serveur
performance_waiters = df_final.groupBy(df_employees.name.alias("waiter_name")).agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.sum("line_total").alias("total_revenue"),
    F.round(F.avg("line_total"), 2).alias("avg_order_value")
).orderBy(F.col("total_revenue").desc())

display(performance_waiters)

Databricks visualization. Run in Databricks to view.

#Exploring Weather table